# Training Data & Features at Scale

Data quality determines model quality — "garbage in, garbage out" applies more to production ML than to any academic setting. This note covers label collection strategies, feature design for recsys/ads, feature stores, and a concrete leakage demo.

## What Interviewers Test
- Implicit vs explicit label tradeoffs and position bias correction
- Feature families for recsys/ads and their freshness requirements
- Feature store architecture: offline vs online, point-in-time correctness
- Feature hashing for large categorical vocabularies
- Sampling strategies for imbalanced or large training logs

## Label Collection Strategies

| Strategy | Signals | Quality | Cost | Scale |
|---|---|---|---|---|
| **Explicit feedback** | Ratings, likes, dislikes | High | High (user friction) | Low |
| **Implicit feedback** | Clicks, dwell-time, shares | Medium | Low (free signal) | High |
| **Human labeling** | Crowd/expert labels | Very high | Very high | Low-medium |
| **Weak supervision** | Heuristic labelers | Medium-low | Low | High |
| **Distillation** | Teacher model labels | Medium-high | Medium | High |

**Position bias:** Items shown at position 1 get clicked more due to position, not quality. Correct with inverse propensity weighting or position-aware models.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

# --- Position bias demo ---
n_queries, n_positions = 1000, 5
# True relevance (position-independent)
true_relevance = np.random.rand(n_queries, n_positions)
# Position propensities: position 1 gets 5x more clicks than position 5
propensities = np.array([0.9, 0.6, 0.4, 0.25, 0.15])
# Observed clicks = relevance × propensity + noise
clicks = (np.random.rand(n_queries, n_positions) < (true_relevance * propensities)).astype(float)

# Naive CTR (biased toward top positions)
naive_ctr  = clicks.mean(axis=0)
# IPW: inverse propensity weighting
ipw_ctr    = (clicks / propensities).mean(axis=0)
true_ctr   = true_relevance.mean(axis=0)

fig, ax = plt.subplots(figsize=(7,3))
ax.plot(range(1,6), naive_ctr, 'b-o', label='Naive CTR (biased)')
ax.plot(range(1,6), ipw_ctr,   'g-s', label='IPW CTR (corrected)')
ax.plot(range(1,6), true_ctr,  'r--', label='True relevance')
ax.set_xlabel('Position'); ax.set_ylabel('Rate')
ax.set_title('Position Bias: Naive vs IPW Correction')
ax.legend(); plt.tight_layout()
plt.savefig('/tmp/position_bias.png', dpi=80); plt.close()

print("Position 1 vs 5 comparison:")
for pos, (n, i, t) in enumerate(zip(naive_ctr, ipw_ctr, true_ctr), 1):
    print(f"  Pos {pos}: naive={n:.3f}, IPW={i:.3f}, true={t:.3f}")


## Feature Families for Recsys/Ads

| Family | Examples | Freshness need | Storage |
|---|---|---|---|
| **User** | Age, gender, location, account age | Daily | Offline |
| **User behavior** | Last 10 clicks, avg session length | Minutes | Online |
| **Item** | Category, price, content embedding | Daily | Offline |
| **Item stats** | Click rate (trailing 7d), freshness | Hours | Offline |
| **Context** | Time of day, device, app version | Real-time | Request |
| **Cross (interaction)** | User × item click history | Minutes | Online |

**Feature hashing:** For categorical features with large vocabularies (user IDs, URLs), hashing avoids a lookup table — `hash(feature) % bucket_size`. Collisions are acceptable if bucket_size >> cardinality.


In [ ]:
# --- Feature hashing ---
def hash_feature(feature, n_buckets=2**16):
    """Map arbitrary string to a bucket index."""
    return hash(str(feature)) % n_buckets

# Simulate item ID encoding
n_items = 1_000_000
item_ids = [f'item_{i}' for i in np.random.randint(0, n_items, 10000)]
buckets = [hash_feature(iid, n_buckets=2**17) for iid in item_ids]

unique_ids = len(set(item_ids))
unique_buckets = len(set(buckets))
collision_rate = 1 - unique_buckets / unique_ids
print(f"Unique items: {unique_ids}")
print(f"Unique buckets (2^17={2**17}): {unique_buckets}")
print(f"Collision rate: {collision_rate:.4f} ({collision_rate*100:.2f}%)")
print("Rule of thumb: n_buckets ≥ 2x expected cardinality keeps collisions < ~40%")


## Feature Stores: Offline vs Online

```
Batch pipeline       Streaming pipeline
(Spark/dbt)          (Flink/Kafka)
     ↓                     ↓
 Offline store ←——— Point-in-time ——→ Online store
 (data lake)          correct join       (Redis/DynamoDB)
                                              ↓
                                        Serving (< 5ms)
```

**Point-in-time correctness:** When training, features must be fetched as of the event timestamp — not the current value. Using future feature values creates a "future leakage" that makes offline metrics look better than online.


In [ ]:
import pandas as pd

# --- Point-in-time correctness demo ---
np.random.seed(42)

# Simulate user feature history (e.g., 7-day active rate updates)
events = pd.DataFrame({
    'user_id': ['u1', 'u1', 'u1', 'u2', 'u2'],
    'timestamp': pd.to_datetime(['2024-01-01', '2024-01-05', '2024-01-10',
                                  '2024-01-03', '2024-01-08']),
    'label': [1, 0, 1, 1, 0]
})

feature_store = pd.DataFrame({
    'user_id': ['u1','u1','u1', 'u2','u2'],
    'feature_ts': pd.to_datetime(['2024-01-01','2024-01-06','2024-01-11',
                                   '2024-01-02','2024-01-09']),
    'active_7d': [0.8, 0.6, 0.9, 0.5, 0.7]  # feature value as of feature_ts
})

# WRONG: use current feature value (data leakage)
latest = feature_store.groupby('user_id').last().reset_index()
wrong = events.merge(latest[['user_id','active_7d']], on='user_id')
print("WRONG join (leakage — uses future feature values):")
print(wrong[['timestamp','user_id','active_7d','label']].to_string(index=False))

# CORRECT: point-in-time join (asof join)
merged_rows = []
for _, ev in events.iterrows():
    hist = feature_store[(feature_store.user_id == ev.user_id) &
                         (feature_store.feature_ts <= ev.timestamp)]
    if len(hist):
        feat_val = hist.iloc[-1]['active_7d']
    else:
        feat_val = np.nan
    merged_rows.append({**ev, 'active_7d': feat_val})
correct = pd.DataFrame(merged_rows)
print("\nCORRECT join (point-in-time — no future leakage):")
print(correct[['timestamp','user_id','active_7d','label']].to_string(index=False))


## Common Interview Questions

**Q: What is position bias and how do you correct for it?**
Items shown in higher positions receive more clicks regardless of quality, because users scan from top to bottom. This biases training data to favor high-position items. Correct with inverse propensity weighting (weight each interaction by 1/propensity of its position) or train a position-aware model that explicitly models position as a feature and removes it at serving time.

**Q: What is a feature store and why do you need one?**
A feature store is a centralized repository that manages the computation, storage, and serving of ML features. It solves three problems: (1) offline/online consistency — same feature logic for training and serving; (2) point-in-time correctness for training; (3) reuse across teams. Without it, teams reimplement the same features with subtle inconsistencies that cause training-serving skew.

**Q: What is training-serving skew?**
Differences between features computed during training and during inference. Causes: different preprocessing code paths, stale features at serving time, missing features handled differently, point-in-time violations in training. Prevention: shared feature computation code, feature monitoring, shadow evaluation.

**Q: How do you handle label delay (e.g., chargeback fraud detected weeks later)?**
Use proxy labels with short delay (e.g., suspicious transaction flags) as early signals. Train on the proxy first, then fine-tune when true labels arrive. Use calibration to align proxy scores with true fraud rates. Monitor for distributional drift as the label distribution shifts when delayed true labels arrive.

## Key Takeaways
- Implicit feedback is abundant but noisy; explicit is sparse but accurate — use both
- Position bias inflates top-position CTR; correct with IPW or position-aware models
- Feature families: user / item / context / cross — each has different freshness needs
- Point-in-time correctness: at training time, join features as of the event timestamp, not current value
- Feature hashing: hash(feature) % n_buckets for large categoricals; 2x cardinality buckets for low collision
- Feature stores provide offline/online consistency, PIT correctness, and cross-team reuse